# Protocol 2.0 — official results review
Read-only analysis of validated aggregates. Export is a separate, opt-in final cell. No model training or prediction generation.


In [ ]:
from pathlib import Path
import os, sys
from io import BytesIO
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/reporting/dashboard_inputs.py').is_file())
if str(root) not in sys.path: sys.path.insert(0,str(root))
from src.reporting.dashboard_inputs import load_batch
from src.reporting.protocol2_dashboard import results_provenance, global_figures, configuration_figure, generate
from IPython.display import display, Image
import matplotlib.pyplot as plt
SOURCE = Path(os.environ.get('PROTOCOL2_BATCH',str(root/'results/dashboard-inputs/67ad3e7/evidence_bundle.zip')))
RESULTS_COMMIT = '67ad3e764dadf13827421ab2d6f78f85809b42b3'
RESULTS_GIT_PATH = 'results/protocol2/windows-official-01/evidence_bundle.zip'
publication = results_provenance(SOURCE,RESULTS_COMMIT,RESULTS_GIT_PATH)
batch = load_batch(SOURCE)
batch.results_commit = publication['results_commit']
SELECTED = [c for c in batch.valid if '-50-LINEAR' in c]
def show(fig):
    buffer=BytesIO(); fig.savefig(buffer,format='png',dpi=150,facecolor='white')
    display(Image(data=buffer.getvalue(),width=1000)); plt.close(fig)
print(batch.batch_id, batch.status)
print('Training:',batch.commit,'Results:',batch.results_commit)
print('254 patients; 3 repeats × 5 outer folds; 24 configurations; 360 completed outer fits.')


## Four pipelines and all-configuration ranking
MM = MinMax; STD = Standard. ANOVA sizes count genes; PCA sizes count components. Ranking is descriptive, not statistical superiority. Sample SD uses three repeat scores, not independent test samples.


In [ ]:
figures = {}
for name,fig in global_figures(batch,SELECTED):
    if name in ('pipeline-heatmap','macro-f1-ranking'): figures[name] = fig
    else: plt.close(fig)
for name in ('pipeline-heatmap','macro-f1-ranking'): show(figures[name])
display(batch.tables['summary.csv'][['configuration','macro_f1_mean','macro_f1_sd','runtime_seconds']].sort_values('macro_f1_mean',ascending=False))


## Subtype overview
Mean repeat-level F1 for each configuration. SS has only 10 patients; MPNST has only 9.


In [ ]:
from src.reporting.protocol2_dashboard import subtype_overview
show(subtype_overview(batch))


## Configuration explorer
Change CONFIGURATION to any of the 24 validated IDs. All three following views use that same selection. The default is the existing workflow reference, not a claim of unbiased winner evaluation.


In [ ]:
CONFIGURATION = 'SVM-STD-ANOVA-150-LINEAR'
for kind in ('subtypes','confusion','stability'):
    show(configuration_figure(batch,CONFIGURATION,kind))


## Runtime, supporting tables and limitations
Three repeats represent 762 prediction events from the same 254 patients, not 762 independent patients. Sample SD is not a confidence interval; mean ± SD can extend outside 0–1. Confusion scales are shared across configurations. Selecting a winner from outer-CV scores introduces selection bias. These results alone do not establish clinical usefulness.


In [ ]:
SHOW_SUPPLEMENT = False
if SHOW_SUPPLEMENT:
    for name,fig in global_figures(batch,SELECTED):
        if name in ('runtime-performance','validation-status','repeat-stability'): show(fig)
        else: plt.close(fig)
    display(batch.overview)
    for filename in ('comparison.csv','reports/repeat_scores.csv','reports/subtype_analysis.csv'):
        frame=batch.tables[filename]
        display(frame[frame.configuration.eq(CONFIGURATION)])


## Optional export — separate from analysis
The cells above do not write report files. Set RUN_EXPORT=True only to produce a new HTML/PNG export. Existing directories and evidence cannot be overwritten.


In [ ]:
from datetime import datetime, timezone
RUN_EXPORT = os.environ.get('PROTOCOL2_EXPORT','0') == '1'
OUTPUT = Path(os.environ.get('PROTOCOL2_REPORT_OUTPUT',str(root/'results/dashboards'/datetime.now(timezone.utc).strftime('redesign-%Y%m%dT%H%M%SZ'))))
if RUN_EXPORT:
    report = generate(SOURCE,OUTPUT,selected=SELECTED,results_commit=RESULTS_COMMIT,results_git_path=RESULTS_GIT_PATH)
    print('Created report.html, report_provenance.json and figures/ in the chosen new directory.')
else:
    print('Export disabled; analysis did not write report files.')
